# accel-sim silicon anchor — optimizer-state precision profiler

Times a manual Adam update (m, v running averages) at **fp32** vs **bf16** state,
for the four parameter counts in GPT-2's own layer shapes — no forward/backward,
just the optimizer arithmetic, isolating exactly what `Optimizer.state_bytes` models.

**Before running:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

Note: Turing (T4, sm75) has no native bf16 arithmetic (that arrived with Ampere) —
PyTorch may promote/emulate bf16 ops here. That's itself useful information, not a
reason to skip the run.

Writes `optimizer_profile.json`, prints it, and auto-downloads it. Bring that file back and run:

```bash
python validate/silicon/compare_optimizer.py optimizer_profile.json
```


In [ ]:
# ---- config (edit if you want) ---------------------------------------------
ITERS  = 50
WARMUP = 15
OUT    = "optimizer_profile.json"

# GPT-2's own shapes -- matches simulator/workloads.py's dimensions.
CHAIN_DIMS = [(768, 768), (768, 768), (768, 3072), (3072, 768)]


In [ ]:
import torch
assert torch.cuda.is_available(), "no CUDA device -- Runtime > Change runtime type > T4 GPU"

device = torch.device("cuda")
gpu = torch.cuda.get_device_name(0)
layer_sizes = [k * n for k, n in CHAIN_DIMS]
print(f"GPU: {gpu}   layer param counts={layer_sizes}   iters={ITERS} (+{WARMUP} warmup)")


In [ ]:
import statistics

def bench_adam_step(n_params, state_dtype, iters, warmup):
    param = torch.randn(n_params, device=device, dtype=torch.float32)
    grad = torch.randn(n_params, device=device, dtype=torch.float32)
    m = torch.zeros(n_params, device=device, dtype=state_dtype)
    v = torch.zeros(n_params, device=device, dtype=state_dtype)
    beta1, beta2, eps, lr = 0.9, 0.999, 1e-8, 1e-4

    def step():
        nonlocal m, v, param
        m = (beta1 * m.float() + (1 - beta1) * grad).to(state_dtype)
        v = (beta2 * v.float() + (1 - beta2) * grad * grad).to(state_dtype)
        param = param - lr * m.float() / (v.float().sqrt() + eps)

    times = []
    for i in range(warmup + iters):
        ev0, ev1 = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        ev0.record()
        step()
        ev1.record()
        torch.cuda.synchronize()
        if i >= warmup:
            times.append(ev0.elapsed_time(ev1))

    return {"mean_ms": statistics.fmean(times),
            "std_ms": statistics.pstdev(times) if len(times) > 1 else 0.0}


In [ ]:
_DTYPES = {"fp32": torch.float32, "bf16": torch.bfloat16}
results = {}
for state_name, state_dtype in _DTYPES.items():
    per_layer = [bench_adam_step(n, state_dtype, ITERS, WARMUP) for n in layer_sizes]
    total_ms = sum(r["mean_ms"] for r in per_layer)
    results[state_name] = {"per_layer": per_layer, "total_ms": total_ms}
    print(f"  {state_name} state   total {total_ms:8.4f} ms   "
          f"per-layer {[round(r['mean_ms'], 4) for r in per_layer]}")

saved = (results["fp32"]["total_ms"] - results["bf16"]["total_ms"]) / results["fp32"]["total_ms"] * 100
print(f"\n  measured saving from bf16 state: {saved:+.1f}%")


In [ ]:
import json, platform

out = {
    "gpu": gpu, "torch": torch.__version__, "cuda": torch.version.cuda,
    "layer_sizes": layer_sizes, "chain_dims": CHAIN_DIMS,
    "iters": ITERS, "warmup": WARMUP,
    "results": results, "host": platform.platform(),
}
with open(OUT, "w") as f:
    json.dump(out, f, indent=2)

print(f"\n===== {OUT} (copy this back if the download fails) =====\n")
print(json.dumps(out, indent=2))

try:
    from google.colab import files
    files.download(OUT)
except Exception as e:
    print(f"\n(auto-download unavailable: {e} -- grab {OUT} from the Files sidebar)")
